In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

In [2]:
df = pd.read_csv("../data/raw/wait_times.csv")
df.head


<bound method NDFrame.head of                     port_name             category lane_type          status  \
0      Bridge of the Americas  Commercial Vehicles   General          closed   
1      Bridge of the Americas  Commercial Vehicles      Fast          closed   
2      Bridge of the Americas   Passenger Vehicles   General        reported   
3      Bridge of the Americas   Passenger Vehicles    Sentri  not applicable   
4      Bridge of the Americas   Passenger Vehicles     Ready        reported   
...                       ...                  ...       ...             ...   
15059                 Stanton   Passenger Vehicles   General          closed   
15060                 Stanton   Passenger Vehicles    Sentri        reported   
15061                 Stanton   Passenger Vehicles     Ready  not applicable   
15062                 Stanton           Pedestrian   General  not applicable   
15063                 Stanton           Pedestrian     Ready  not applicable   

       wa

In [3]:
pd.to_numeric(df['wait_minutes'], errors='coerce').notnull()

0        False
1        False
2         True
3        False
4         True
         ...  
15059    False
15060     True
15061    False
15062    False
15063    False
Name: wait_minutes, Length: 15064, dtype: bool

In [4]:
df_clean = df[pd.to_numeric(df['wait_minutes'], errors='coerce').notnull()]
df['status'].value_counts()

status
reported          6782
not applicable    5391
closed            2891
Name: count, dtype: int64

Reported: 6782

NA: 5391

Closed: 2891

In [5]:
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
df_clean.head(1000)

,port_name,category,lane_type,status,wait_minutes,timestamp
2,Bridge of the Americas,Passenger Vehicles,General,reported,50.0,2026-08-09 15:26:14.176934
4,Bridge of the Americas,Passenger Vehicles,Ready,reported,45.0,2026-08-09 15:26:14.176934
5,Bridge of the Americas,Pedestrian,General,reported,1.0,2026-08-09 15:26:14.176934
6,Bridge of the Americas,Pedestrian,Ready,reported,1.0,2026-08-09 15:26:14.176934
9,Paso Del Norte,Passenger Vehicles,General,reported,54.0,2026-08-09 15:26:14.176934
...,...,...,...,...,...,...
1970,Paso Del Norte,Passenger Vehicles,Sentri,reported,1.0,2026-08-14 19:40:48.703565
1971,Paso Del Norte,Passenger Vehicles,Ready,reported,30.0,2026-08-14 19:40:48.703565
1972,Paso Del Norte,Pedestrian,General,reported,8.0,2026-08-14 19:40:48.703565
1973,Paso Del Norte,Pedestrian,Ready,reported,5.0,2026-08-14 19:40:48.703565


In [6]:
df_clean['hour'] = df_clean['timestamp'].dt.hour
df_clean['day_of_week'] = df_clean['timestamp'].dt.day_of_week

df_clean.head(50)

,port_name,category,lane_type,status,wait_minutes,timestamp,hour,day_of_week
2,Bridge of the Americas,Passenger Vehicles,General,reported,50.0,2026-08-09 15:26:14.176934,15,6
4,Bridge of the Americas,Passenger Vehicles,Ready,reported,45.0,2026-08-09 15:26:14.176934,15,6
5,Bridge of the Americas,Pedestrian,General,reported,1.0,2026-08-09 15:26:14.176934,15,6
6,Bridge of the Americas,Pedestrian,Ready,reported,1.0,2026-08-09 15:26:14.176934,15,6
9,Paso Del Norte,Passenger Vehicles,General,reported,54.0,2026-08-09 15:26:14.176934,15,6
10,Paso Del Norte,Passenger Vehicles,Sentri,reported,5.0,2026-08-09 15:26:14.176934,15,6
11,Paso Del Norte,Passenger Vehicles,Ready,reported,50.0,2026-08-09 15:26:14.176934,15,6
12,Paso Del Norte,Pedestrian,General,reported,8.0,2026-08-09 15:26:14.176934,15,6
13,Paso Del Norte,Pedestrian,Ready,reported,5.0,2026-08-09 15:26:14.176934,15,6
17,Ysleta,Passenger Vehicles,Sentri,reported,1.0,2026-08-09 15:26:14.176934,15,6


In [7]:
df_no_dupes2_date = df['timestamp'].dt.date.rename("date")
df_no_dupes2 = df.groupby(df_no_dupes2_date).nunique()
df_no_dupes2


AttributeError: Can only use .dt accessor with datetimelike values

In [ ]:
df['timestamp'].dtype
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['timestamp'].head(10)

0   2026-08-09 15:26:14.176934
1   2026-08-09 15:26:14.176934
2   2026-08-09 15:26:14.176934
3   2026-08-09 15:26:14.176934
4   2026-08-09 15:26:14.176934
5   2026-08-09 15:26:14.176934
6   2026-08-09 15:26:14.176934
7   2026-08-09 15:26:14.176934
8   2026-08-09 15:26:14.176934
9   2026-08-09 15:26:14.176934
Name: timestamp, dtype: datetime64[us]

Implements cyclical feature encoding for both hour and day_of_week columns

In [ ]:
df_clean.index
df_clean['hour_sin'] = np.sin(2 * np.pi * df_clean['hour'] / 24)
df_clean['hour_cos'] = np.cos(2 * np.pi * df_clean['hour'] / 24)

df_clean['day_of_week_sin'] = np.sin(2 * np.pi * df_clean['day_of_week'] / 24)
df_clean['day_of_week_cos'] = np.cos(2 * np.pi * df_clean['day_of_week'] / 24)

df_clean



,port_name,category,lane_type,status,wait_minutes,timestamp,hour,day_of_week,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos
2,Bridge of the Americas,Passenger Vehicles,General,reported,50.0,2026-08-09 15:26:14.176934,15,6,-0.707107,-0.707107,1.0,6.123234e-17
4,Bridge of the Americas,Passenger Vehicles,Ready,reported,45.0,2026-08-09 15:26:14.176934,15,6,-0.707107,-0.707107,1.0,6.123234e-17
5,Bridge of the Americas,Pedestrian,General,reported,1.0,2026-08-09 15:26:14.176934,15,6,-0.707107,-0.707107,1.0,6.123234e-17
6,Bridge of the Americas,Pedestrian,Ready,reported,1.0,2026-08-09 15:26:14.176934,15,6,-0.707107,-0.707107,1.0,6.123234e-17
9,Paso Del Norte,Passenger Vehicles,General,reported,54.0,2026-08-09 15:26:14.176934,15,6,-0.707107,-0.707107,1.0,6.123234e-17
...,...,...,...,...,...,...,...,...,...,...,...,...
15051,Ysleta,Commercial Vehicles,Fast,reported,5.0,2026-09-28 14:38:18.905090,14,0,-0.500000,-0.866025,0.0,1.000000e+00
15053,Ysleta,Passenger Vehicles,Sentri,reported,1.0,2026-09-28 14:38:18.905090,14,0,-0.500000,-0.866025,0.0,1.000000e+00
15054,Ysleta,Passenger Vehicles,Ready,reported,25.0,2026-09-28 14:38:18.905090,14,0,-0.500000,-0.866025,0.0,1.000000e+00
15055,Ysleta,Pedestrian,General,reported,5.0,2026-09-28 14:38:18.905090,14,0,-0.500000,-0.866025,0.0,1.000000e+00


Encodes Boolean values for port_name, category, and lane_type

In [ ]:
df_encoded = pd.get_dummies(df_clean, columns=['port_name', 'category', 'lane_type'])

df_encoded.columns.tolist()


['status',
 'wait_minutes',
 'timestamp',
 'hour',
 'day_of_week',
 'hour_sin',
 'hour_cos',
 'day_of_week_sin',
 'day_of_week_cos',
 'port_name_Bridge of the Americas',
 'port_name_Paso Del Norte',
 'port_name_Stanton',
 'port_name_Ysleta',
 'category_Commercial Vehicles',
 'category_Passenger Vehicles',
 'category_Pedestrian',
 'lane_type_Fast',
 'lane_type_General',
 'lane_type_Ready',
 'lane_type_Sentri']

Use Boolean Masking on df_encoded to split into a train and test set

In [ ]:
cutoff = df_clean['timestamp'].quantile(0.8)
mask = df_clean['timestamp'] <= cutoff

df_train = df_encoded[mask]
df_test = df_encoded[~mask]

df_train_unencoded = df_clean[mask]
df_test_unencoded = df_clean[~mask]

len(df_train) / len(df_encoded)

0.8012385726924212

In [ ]:
X_train = df_train.drop(columns=['wait_minutes'])
y_train = df_train['wait_minutes']

X_test = df_test.drop(columns=['wait_minutes'])
y_test = df_test['wait_minutes']
print(X_train.shape[1])
print(y_train.shape[0])

15
5434


In [ ]:
lookup = df_train_unencoded.groupby(['port_name', 'hour'])['wait_minutes'].mean().reset_index()
lookup

,port_name,hour,wait_minutes
0,Bridge of the Americas,0,33.189189
1,Bridge of the Americas,1,25.125000
2,Bridge of the Americas,2,30.160000
3,Bridge of the Americas,3,31.672727
4,Bridge of the Americas,4,27.162791
...,...,...,...
85,Ysleta,19,24.719101
86,Ysleta,20,22.520408
87,Ysleta,21,22.121495
88,Ysleta,22,22.434343


In [ ]:
df_test_unencoded.groupby(['port_name', 'hour'])['wait_minutes'].mean()

port_name               hour
Bridge of the Americas  0       30.500000
                        1       36.235294
                        2       10.500000
                        3       29.250000
                        5       13.750000
                                  ...    
Ysleta                  19      20.361111
                        20      22.900000
                        21      24.393939
                        22      14.400000
                        23      23.511628
Name: wait_minutes, Length: 87, dtype: float64

In [ ]:
df_unencoded_merge = pd.merge(df_test_unencoded, lookup, on=['port_name', 'hour'], how='left', suffixes=['_actual', '_predicted'])
print(df_unencoded_merge)



merged_clean = df_unencoded_merge.dropna(subset=['wait_minutes_predicted'])

                   port_name             category lane_type    status  \
0     Bridge of the Americas  Commercial Vehicles   General  reported   
1     Bridge of the Americas  Commercial Vehicles      Fast  reported   
2     Bridge of the Americas   Passenger Vehicles   General  reported   
3     Bridge of the Americas   Passenger Vehicles     Ready  reported   
4     Bridge of the Americas           Pedestrian   General  reported   
...                      ...                  ...       ...       ...   
1343                  Ysleta  Commercial Vehicles      Fast  reported   
1344                  Ysleta   Passenger Vehicles    Sentri  reported   
1345                  Ysleta   Passenger Vehicles     Ready  reported   
1346                  Ysleta           Pedestrian   General  reported   
1347                 Stanton   Passenger Vehicles    Sentri  reported   

      wait_minutes_actual                  timestamp  hour  day_of_week  \
0                    20.0 2026-09-11 18:48:39.53

Use an MAE to establish a baseline for the other models to beat.

In [ ]:
baseline_mae = sklearn.metrics.mean_absolute_error(merged_clean['wait_minutes_actual'], merged_clean['wait_minutes_predicted'])
baseline_mae

19.030476755389476

In [ ]:
linear_regression_model = LinearRegression()
linear_regression_model.fit(X_train, y_train)
predictions = linear_regression_model.predict(X_test)


In [ ]:
linear_regression_mae = mean_absolute_error(y_test, predictions)
print(linear_regression_mae)